# 03 - Behavioural Cloning (M3, M4, M5, M8, RQ1, RQ3, RQ4)

**Group members:** TODO

Library BC vs from-scratch BC, dataset-size ablation, architecture sweep, and offline-vs-online metric divergence.

In [ ]:
# Make src importable whether run from notebooks/ or project root
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
# On Colab: mount Drive and set PROJECT_DATA_ROOT before importing src
from src import config, seeding, envs, collect, eval, plotting
seeding.set_seed(0)
ENV_ID = 'Walker2d-v4'  # switch to 'Ant-v4' for the second environment
print('device', config.device(), '| env', ENV_ID)

In [ ]:
data = collect.load(config.DATA_DIR / ENV_ID)
expert_mean = float(data['episode_returns'].mean())
DEVICE = config.device()

## M3/M4 - Library BC (imitation) vs from-scratch BC

Hold epochs / lr / batch / seed equal for a fair comparison.

In [ ]:
from src import bc_bridge, bc_scratch
# Library version
trainer, env = bc_bridge.train_bc_imitation(data['observations'], data['actions'],
                                            ENV_ID, n_epochs=50, device=DEVICE)
lib_mean, lib_std = eval.evaluate(trainer.policy, ENV_ID)
# From-scratch version
student, hist = bc_scratch.train_bc(data['observations'], data['actions'],
                                    n_epochs=50, device=DEVICE)
scr_mean, scr_std = eval.evaluate_torch(student, ENV_ID, DEVICE)
print('library  %.1f +/- %.1f' % (lib_mean, lib_std))
print('scratch  %.1f +/- %.1f' % (scr_mean, scr_std))
plotting.save(plotting.learning_curves(hist['train'], hist['val']),
              config.OUTPUTS_DIR / f'bc_learning_curves_{ENV_ID}.png')

## M5 - Dataset-size ablation (RQ2)

{5, 10, 20, 50, 100} episodes x 5 seeds; mean +/- std with error bars.

In [ ]:
import numpy as np
episode_counts, results = [5, 10, 20, 50, 100], {}
for n_ep in episode_counts:
    sub_obs, sub_acts = collect.subset(data, n_ep)
    runs = []
    for seed in config.SEEDS:
        s, _ = bc_scratch.train_bc(sub_obs, sub_acts, seed=seed, n_epochs=50, device=DEVICE)
        m, _ = eval.evaluate_torch(s, ENV_ID, DEVICE)
        runs.append(m)
    results[n_ep] = runs
    print('n_ep=%3d mean=%.1f std=%.1f' % (n_ep, np.mean(runs), np.std(runs)))
means = [np.mean(results[n]) for n in episode_counts]
stds  = [np.std(results[n]) for n in episode_counts]
plotting.save(plotting.ablation(episode_counts, means, stds, expert_mean),
              config.OUTPUTS_DIR / f'bc_ablation_data_size_{ENV_ID}.png')

## M8 - Architecture sweep (RQ4)

Small MLP, default, large MLP, skip-connections. Does capacity change the gap to the expert?

In [ ]:
archs = {'small': (64, 64), 'default': (256, 256), 'large': (512, 512)}
for name, hidden in archs.items():
    s, _ = bc_scratch.train_bc(data['observations'], data['actions'],
                               hidden=hidden, n_epochs=50, device=DEVICE)
    m, sd = eval.evaluate_torch(s, ENV_ID, DEVICE)
    print('%-8s %.1f +/- %.1f' % (name, m, sd))
# skip-connection variant: bc_scratch.train_bc(..., skip=True)

## RQ3 - Does lower BC loss imply a better policy?

Compare validation MSE against environment return across epochs and discuss.